<img src="logo.png" alt="Vegeta" width="240">

# Unmanned ground vehicle — a small rover for rugged terrain, mechanically

A 3 kg four-wheel rover with trailing-arm suspension, printed in PA12-CF, for rocky trails. The
notebook covers the mechanics end to end:

```
CAD (Dedalus): chassis tub, four trailing arms, wheels ─► masses, wheel loads at rest
terrain profiles (ISO 8608 classes + rocks + a drop) ─► quarter-car dynamics (numpy) ─► wheel force histories
peak loads ─► static FEA of the arm and the chassis in torsion (Talos), arm modes with the wheel mass
force histories ─► rainflow (Chronos) ─► spectra for three terrains ─► fatigue on the arm's stress field
fleet usage ─► life ─► print the arm (Mellonia)
```

Every input is explicit and coarse (spring rates, tyre stiffness, an assumed S-N curve). Compare,
record, then test on a real trail.

In [ ]:
import json, math, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from vegeta import dedalus, talos, chronos, mellonia
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.mellonia import viz as mviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM

RUNS = Path("_runs/rover"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
design_file = RUNS / "rover.py"
shutil.copy(Path("designs/rover.py"), design_file)
rover_design = dedalus.load_design(f"{design_file}:Rover")
pd.DataFrame(rover_design.params.table()).set_index("name")

## 1. The vehicle

In [ ]:
p = rover_design.resolve()
rover = rover_design.generate()
arm = rover_design.generate(part="arm")
chassis = rover_design.generate(part="chassis")
cad = {k: g.export(RUNS / "cad" / k, stl_tolerance=0.05) for k, g in (("rover", rover), ("arm", arm), ("chassis", chassis))}
dviz.show(dviz.plot3d(rover))

In [ ]:
fig = dviz.plot_sections(rover, normal="y", positions=[0.0, p["width"] / 2 + 8, p["width"] / 2 + 45], cols=3)   # centre, arm plane, wheel plane
fig = dviz.plot_sections(arm, normal="x", positions=[10.0, 45.0, 85.0], cols=3)

In [ ]:
RHO_PA12CF = 1.1e-3       # g/mm^3
parts = pd.DataFrame([
    ("chassis (PA12-CF, from CAD)", chassis.volume * RHO_PA12CF),
    ("suspension arms 4x (from CAD)", 4 * arm.volume * RHO_PA12CF),
    ("wheels + tyres 4x", 4 * 120.0),
    ("motors + gearboxes 4x", 4 * 95.0),
    ("battery 3S 5000 mAh", 380.0),
    ("controller, radio, camera", 180.0),
    ("wiring, bolts, bearings", 120.0),
], columns=["part", "mass_g"]).set_index("part")
M_TOTAL = parts["mass_g"].sum() / 1000
M_WHEEL = (120.0 + 95.0 * 0.5) / 1000                  # unsprung mass per corner: wheel + half the motor/gearbox
M_SPRUNG_CORNER = (M_TOTAL - 4 * M_WHEEL) / 4
G = 9.81
print(f"total {M_TOTAL:.2f} kg | sprung per corner {M_SPRUNG_CORNER:.3f} kg | static wheel load {M_TOTAL * G / 4:.1f} N")
parts.round(1)

## 2. Terrain and wheel loads: a quarter-car model over three terrains

Each corner is a two-mass system: the sprung quarter of the body on the suspension (a torsion spring
at the arm pivot, expressed as a vertical rate at the wheel, with a damper) and the wheel on its tyre
stiffness, driven by the ground profile under the wheel. Profiles are ISO 8608 classes generated
from their PSD (seeded), with rocks (half-sine bumps) and a drop added for the rough missions.
Tyre force below zero means the wheel left the ground.

In [ ]:
K_SUSP = 520.0       # N/m at the wheel (torsion spring at the pivot), sag ~ 14 mm under the static load
ZETA = 0.3           # damping ratio of the suspension (damper or friction at the pivot)
K_TYRE = 6000.0      # N/m, foam-filled tyre
C_SUSP = 2 * ZETA * math.sqrt(K_SUSP * M_SPRUNG_CORNER)

def iso8608_profile(length_m, dx, gd_n0, seed, n0=0.1, n_min=0.02, n_max=8.0, n_waves=400):
    # displacement PSD Gd(n) = Gd(n0) (n/n0)^-2, superposition of sinusoids with random phase
    rng = np.random.default_rng(seed)
    x = np.arange(0, length_m, dx)
    ns = np.linspace(n_min, n_max, n_waves)
    dn = ns[1] - ns[0]
    amps = np.sqrt(2 * gd_n0 * (ns / n0) ** -2 * dn)
    phases = rng.uniform(0, 2 * math.pi, n_waves)
    z = (amps[None, :] * np.sin(2 * math.pi * ns[None, :] * x[:, None] + phases[None, :])).sum(axis=1)
    return x, z

def add_rocks(x, z, height_m, width_m, spacing_m, seed):
    rng = np.random.default_rng(seed)
    z = z.copy()
    centres = np.arange(spacing_m, x[-1] - spacing_m, spacing_m)
    for xc in centres + rng.uniform(-0.3, 0.3, len(centres)):
        mask = np.abs(x - xc) < width_m / 2
        z[mask] += height_m * np.cos(math.pi * (x[mask] - xc) / width_m)
    return z

def add_drop(x, z, at_m, depth_m):
    z = z.copy(); z[x > at_m] -= depth_m
    return z

def quarter_car(x, z_road, speed, dt=5e-4):
    # states: sprung z_s, wheel z_w (relative to static equilibrium), velocities; ground z_r(t) = z_road(x = v t)
    t = np.arange(0, x[-1] / speed, dt)
    zr = np.interp(t * speed, x, z_road)
    zs = zw = vs = vw = 0.0
    Fs, Ft, As, travel = np.zeros_like(t), np.zeros_like(t), np.zeros_like(t), np.zeros_like(t)
    for i, z_r in enumerate(zr):
        f_susp = K_SUSP * (zw - zs) + C_SUSP * (vw - vs)                     # suspension force on the body (+ up)
        f_tyre = max(K_TYRE * (z_r - zw), -M_TOTAL * G / 4)                  # tyre can only push (down to zero contact)
        a_s = f_susp / M_SPRUNG_CORNER
        a_w = (f_tyre - f_susp) / M_WHEEL
        vs += a_s * dt; vw += a_w * dt; zs += vs * dt; zw += vw * dt
        Fs[i], Ft[i], As[i], travel[i] = f_susp + M_SPRUNG_CORNER * G, f_tyre + M_TOTAL * G / 4, a_s, zw - zs
    return t, zr, Fs, Ft, As, travel

terrains = {
    "paved patrol":  {"class": "A", "gd": 16e-6,   "speed": 1.5, "length": 300.0, "rocks": None,                "drop": None,        "seed": 1},
    "gravel trail":  {"class": "C", "gd": 256e-6,  "speed": 1.2, "length": 300.0, "rocks": (0.02, 0.10, 6.0),   "drop": None,        "seed": 2},
    "rocky field":   {"class": "E", "gd": 4096e-6, "speed": 0.8, "length": 200.0, "rocks": (0.04, 0.12, 2.5),   "drop": (150.0, 0.12), "seed": 3},
}
sims = {}
for name, tr in tqdm(terrains.items(), desc="terrains"):
    x, z = iso8608_profile(tr["length"], 0.005, tr["gd"], tr["seed"])
    if tr["rocks"]:
        z = add_rocks(x, z, *tr["rocks"], seed=tr["seed"] + 10)
    if tr["drop"]:
        z = add_drop(x, z, *tr["drop"])
    t, zr, Fs, Ft, As, travel = quarter_car(x, z, tr["speed"])
    sims[name] = dict(t=t, zr=zr, Fs=Fs, Ft=Ft, As=As, travel=travel, duration_s=t[-1], speed=tr["speed"], length=tr["length"])
summary = pd.DataFrame({k: {"speed_m_s": s["speed"], "duration_s": s["duration_s"], "tyre_force_max_N": s["Ft"].max(),
                            "tyre_force_min_N": s["Ft"].min(), "arm_force_max_N": s["Fs"].max(),
                            "body_accel_rms_g": np.sqrt(np.mean(s["As"] ** 2)) / G, "travel_max_mm": np.abs(s["travel"]).max() * 1000,
                            "airborne_%": 100 * np.mean(s["Ft"] <= 1e-6)} for k, s in sims.items()}).T.round(2)
summary

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 9))
for (name, s), (a1, a2) in zip(sims.items(), axes):
    a1.plot(s["t"] * s["speed"], s["zr"] * 1000, lw=0.6, color="#5a6a7a"); a1.set(title=f"{name}: ground profile", xlabel="distance [m]", ylabel="height [mm]")
    a2.plot(s["t"], s["Ft"], lw=0.5, label="tyre force"); a2.plot(s["t"], s["Fs"], lw=0.5, label="arm (suspension) force")
    a2.set(title=f"{name}: wheel forces", xlabel="time [s]", ylabel="N"); a2.legend(fontsize=8)
    for a in (a1, a2): a.grid(alpha=0.3)
fig.tight_layout()

The chassis feels the arm force; the arm feels the tyre force at its axle and, at every rock edge,
a longitudinal component ≈ vertical force × local slope. That longitudinal history is derived from the
profile slope where the tyre is loaded.

In [ ]:
for name, s in sims.items():
    slope = np.gradient(s["zr"], s["t"] * s["speed"])
    s["Fx"] = np.clip(s["Ft"] * np.clip(np.abs(slope), 0, 1.5), 0, None)     # rock strikes: longitudinal load on the arm
print({k: round(float(s["Fx"].max()), 1) for k, s in sims.items()}, "N peak longitudinal")

## 3. Static strength of the parts under the operating cases (Talos)

Four operating situations, each an explicit engineer input, turned into wheel loads and then into
load cases on three elements — the suspension arm (FEA), the chassis (FEA: torsion and plate bending)
and the steel pins (hand calculation):

| case | what it means for the loads |
|---|---|
| level ground | static wheel load `M g / 4`; the rocky-field peak from the quarter-car model as the dynamic case |
| hill climbing (`GRADE_DEG`) | weight shifts to the rear axle by `h_cg / wheelbase × tan θ`; each wheel pushes with the traction share of `M g sin θ` plus rolling, capped by tyre friction and by the motor stall torque |
| mud | rolling resistance `ROLL_MUD` (sinkage), traction capped by `MU_MUD`; a wheel dragged sideways in a rut sees a lateral force; stuck wheels see the full stall torque |
| heavy payload (`PAYLOAD_KG` on the plate) | every load scales with the mass; the plate bends under the payload; the quarter-car model is re-run with the extra sprung mass for the rocky field |

In [ ]:
GRADE_DEG = 30.0            # steepest slope to climb [deg] (input)
MU_DRY, MU_MUD = 0.8, 0.35  # tyre friction on rock and in mud (inputs)
ROLL_MUD = 0.30             # rolling-resistance coefficient in soft mud (sinkage; input)
PAYLOAD_KG = 6.0            # heavy load carried on the plate (input)
H_CG = 0.09                 # centre of gravity height above ground [m] (input)
T_STALL = 1.2               # stall torque at the wheel, motor x gearbox [N m] (input)
R_WHEEL = p["wheel_diameter"] / 2000
WHEELBASE = 2 * (p["length"] / 2 - p["pivot_x"]) / 1000
PETG_STEEL_PIN = talos.Material("steel pin (C45)", youngs_modulus=210000.0, poissons_ratio=0.30, density=7.85e-9, yield_strength=400.0, source="handbook")

def wheel_loads(mass, grade_deg, roll, mu):
    W, th = mass * G, math.radians(grade_deg)
    shift = H_CG / WHEELBASE * math.tan(th)
    rear, front = W * math.cos(th) * (0.5 + shift) / 2, W * math.cos(th) * (0.5 - shift) / 2
    need = (W * math.sin(th) + roll * W * math.cos(th)) / 4                      # traction per wheel to hold the climb
    trac_rear = min(need, mu * rear, T_STALL / R_WHEEL)
    return {"rear_N": rear, "front_N": front, "traction_needed_N": need, "traction_rear_N": trac_rear,
            "can_climb": trac_rear * 2 + min(need, mu * front, T_STALL / R_WHEEL) * 2 >= W * math.sin(th) + roll * W * math.cos(th)}
M_LOADED = M_TOTAL + PAYLOAD_KG
level, hill, hill_loaded = wheel_loads(M_TOTAL, 0, 0.02, MU_DRY), wheel_loads(M_TOTAL, GRADE_DEG, 0.02, MU_DRY), wheel_loads(M_LOADED, GRADE_DEG, 0.02, MU_DRY)
mud, mud_loaded = wheel_loads(M_TOTAL, 0, ROLL_MUD, MU_MUD), wheel_loads(M_LOADED, 0, ROLL_MUD, MU_MUD)
sag_mm = PAYLOAD_KG * G / 4 / K_SUSP * 1000
print(f"hill {GRADE_DEG:.0f} deg: rear wheel {hill['rear_N']:.0f} N, front {hill['front_N']:.0f} N, traction needed {hill['traction_needed_N']:.1f} N/wheel -> "
      f"{'climbs' if hill['can_climb'] else 'CANNOT climb'} empty, {'climbs' if hill_loaded['can_climb'] else 'CANNOT climb'} with {PAYLOAD_KG:.0f} kg")
print(f"mud: traction needed {mud['traction_needed_N']:.1f} N/wheel vs available {mud['traction_rear_N']:.1f} N (mu {MU_MUD}, stall {T_STALL / R_WHEEL:.1f} N) -> "
      f"{'moves' if mud['can_climb'] else 'STUCK'} empty, {'moves' if mud_loaded['can_climb'] else 'STUCK'} loaded")
print(f"payload {PAYLOAD_KG:.0f} kg: suspension sag {sag_mm:.0f} mm more per corner (spring {K_SUSP:.0f} N/m); check against the travel of the real arm")
peak_z = max(s["Ft"].max() for s in sims.values())                        # rocky-field peaks, empty
peak_x = max(s["Fx"].max() for s in sims.values())
# the rocky field again with the payload on board (the quarter-car model reads the global masses)
_saved = (M_TOTAL, M_SPRUNG_CORNER, C_SUSP)
M_TOTAL, M_SPRUNG_CORNER = M_LOADED, (M_LOADED - 4 * M_WHEEL) / 4
C_SUSP = 2 * ZETA * math.sqrt(K_SUSP * M_SPRUNG_CORNER)
tr = terrains["rocky field"]
x_, z_ = iso8608_profile(tr["length"], 0.005, tr["gd"], tr["seed"]); z_ = add_drop(x_, add_rocks(x_, z_, *tr["rocks"], seed=tr["seed"] + 10), *tr["drop"])
t_, zr_, Fs_l, Ft_l, As_l, travel_l = quarter_car(x_, z_, tr["speed"])
M_TOTAL, M_SPRUNG_CORNER, C_SUSP = _saved
peak_z_loaded, peak_x_loaded = Ft_l.max(), (Ft_l * np.clip(np.abs(np.gradient(zr_, t_ * tr["speed"])), 0, 1.5)).max()
print(f"rocky field with the payload: peak tyre force {peak_z_loaded:.0f} N (vs {peak_z:.0f} N empty), travel {np.abs(travel_l).max() * 1000:.0f} mm")

In [ ]:
PA12CF = talos.Material("PA12-CF (printed)", youngs_modulus=3500.0, poissons_ratio=0.40, density=1.1e-9, yield_strength=60.0,
                        source="nominal datasheet, flat orientation; assume 30 % less across layers")
L, h = p["arm_length"], p["arm_height"]
rp, ra = p["pivot_diameter"] / 2 + 0.2, p["axle_diameter"] / 2 + 0.2
ARM_REGIONS = [talos.SurfacesInBox("pivot", (-rp, -h, -rp, rp, h, rp)), talos.SurfacesInBox("axle", (L - ra, -h, -ra, L + ra, h, ra))]
WHEEL_MASS_T = M_WHEEL * 1e-3                                                  # tonnes at the axle

def arm_model(loads, name, masses=()):
    return talos.StructuralModel(cad["arm"].artifacts["step"], "mm-N-MPa", PA12CF, ARM_REGIONS, [talos.FixedSupport("pivot")], loads,
                                 talos.MeshSettings(element_size=2.0), name=name, masses=list(masses))

peak_z = max(s["Ft"].max() for s in sims.values())
peak_x = max(s["Fx"].max() for s in sims.values())
# the arm's load cases: fz up at the axle (tyre force), fx longitudinal (traction / rock strike), fy lateral (sideways drag in mud)
ARM_CASES = {
    "level: rocky-field peak": dict(fz=peak_z, fx=peak_x),
    "hill climbing, rear wheel": dict(fz=hill["rear_N"], fx=hill["traction_rear_N"]),
    "mud: dragged sideways": dict(fz=mud["rear_N"], fx=mud["traction_rear_N"], fy=0.5 * mud["rear_N"]),
    "mud: stuck at stall torque": dict(fz=mud["rear_N"], fx=T_STALL / R_WHEEL),
    "heavy payload: rocky-field peak": dict(fz=peak_z_loaded, fx=peak_x_loaded),
    "heavy payload: hill climbing": dict(fz=hill_loaded["rear_N"], fx=hill_loaded["traction_rear_N"]),
}
arm_model([talos.Force("axle", fz=1.0)], "mesh").mesh(RUNS / "arm_mesh", progress=True)      # a placeholder load: only the mesh is used
arm_results = {}
for name, f in tqdm(ARM_CASES.items(), desc="arm cases"):
    d = RUNS / f"arm_{name.split(':')[0].replace(' ', '_')}_{len(arm_results)}"
    shutil.copytree(RUNS / "arm_mesh", d, dirs_exist_ok=True)
    arm_results[name] = arm_model([talos.Force("axle", **f)], name).solve(d)
res_arm = arm_results["level: rocky-field peak"]
arm_peak = arm_model([talos.Force("axle", **ARM_CASES["level: rocky-field peak"])], "arm_peak")
tviz.show(tviz.plot_problem(arm_peak, RUNS / "arm_mesh" / "mesh.msh"))
worst_arm = max(arm_results, key=lambda k: arm_results[k].metrics.get("max_von_mises", 0))
print("worst arm case:", worst_arm)
tviz.show(tviz.plot_results(arm_results[worst_arm], field="von_mises"))

In [ ]:
px, pz = p["length"] / 2 - p["pivot_x"], -p["plate_thickness"] - p["rail_height"] / 2
rr = p["pivot_diameter"] / 2 + 0.3
CH_REGIONS = [talos.SurfacesInBox(f"pivot_{'f' if sx > 0 else 'r'}{'l' if sy > 0 else 'r'}",
                                  (sx * px - rr, sy * p["width"] / 2 - 10, pz - rr, sx * px + rr, sy * p["width"] / 2 + 10, pz + rr))
              for sx in (-1, 1) for sy in (-1, 1)] + [talos.SurfacesOnPlane("plate_top", "z", 0.0)]
arm_peak_force = max(s["Fs"].max() for s in sims.values())
def chassis_model(supports, loads, name):
    return talos.StructuralModel(cad["chassis"].artifacts["step"], "mm-N-MPa", PA12CF, CH_REGIONS, supports, loads,
                                 talos.MeshSettings(element_size=6.0), name=name)
chassis_model([talos.FixedSupport("pivot_rl")], [talos.Force("pivot_fl", fz=1.0)], "mesh").mesh(RUNS / "chassis_mesh", progress=True)
ALL_PIVOTS = [talos.FixedSupport(f"pivot_{k}") for k in ("fl", "fr", "rl", "rr")]
CH_CASES = {
    "torsion: one wheel on a rock, the opposite in a hole": ([talos.FixedSupport("pivot_rl"), talos.FixedSupport("pivot_fr")],
                                                            [talos.Force("pivot_fl", fz=-arm_peak_force), talos.Force("pivot_rr", fz=-arm_peak_force)]),
    "heavy payload on the plate": (ALL_PIVOTS, [talos.Pressure("plate_top", PAYLOAD_KG * G / (p["length"] * p["width"]) * 2.0)]),   # x2: a bump at 2 g
    "hill climbing, loaded: traction at the pivots": ([talos.FixedSupport("pivot_fl"), talos.FixedSupport("pivot_fr")],
                                                     [talos.Force("pivot_rl", fx=-hill_loaded["traction_rear_N"], fz=-hill_loaded["rear_N"]),
                                                      talos.Force("pivot_rr", fx=-hill_loaded["traction_rear_N"], fz=-hill_loaded["rear_N"]),
                                                      talos.Pressure("plate_top", PAYLOAD_KG * G / (p["length"] * p["width"]))]),
}
ch_results = {}
for name, (sup, loads) in tqdm(CH_CASES.items(), desc="chassis cases"):
    d = RUNS / f"chassis_{len(ch_results)}"
    shutil.copytree(RUNS / "chassis_mesh", d, dirs_exist_ok=True)
    ch_results[name] = chassis_model(sup, loads, name).solve(d)
res_ch = ch_results["torsion: one wheel on a rock, the opposite in a hole"]
worst_ch = max(ch_results, key=lambda k: ch_results[k].metrics.get("max_von_mises", 0))
print("worst chassis case:", worst_ch)
tviz.show(tviz.plot_results(ch_results[worst_ch], field="von_mises"))

In [ ]:
# steel pins by hand: the axle as a cantilever from the arm to the wheel centre, the pivot pin bending over the rail-to-arm gap (double shear)
def pin_bending(F, lever_mm, d_mm): return F * lever_mm / (math.pi * d_mm**3 / 32)
lever_axle = p["wheel_offset"] + p["wheel_width"] / 2
lever_pivot = (p["arm_width"] / 2 + 3.0) / 2
pins = {}
for name, f in ARM_CASES.items():
    F = math.sqrt(f.get("fz", 0) ** 2 + f.get("fx", 0) ** 2 + f.get("fy", 0) ** 2)
    pins[name] = {"resultant_N": F, "axle_bending_MPa": pin_bending(F, lever_axle, p["axle_diameter"]), "pivot_pin_bending_MPa": pin_bending(F, lever_pivot, p["pivot_diameter"]),
                  "pin_SF": PETG_STEEL_PIN.yield_strength / max(pin_bending(F, lever_axle, p["axle_diameter"]), pin_bending(F, lever_pivot, p["pivot_diameter"]))}
static = pd.concat([
    pd.DataFrame({f"arm — {k}": {"load": ", ".join(f"{kk} {vv:.0f} N" for kk, vv in ARM_CASES[k].items()), "max_von_mises_MPa": r.metrics.get("max_von_mises"),
                                 "SF_yield": r.metrics.get("safety_factor_yield"), "deflection_mm": r.metrics.get("max_displacement")} for k, r in arm_results.items()}).T,
    pd.DataFrame({f"chassis — {k}": {"load": "see the case", "max_von_mises_MPa": r.metrics.get("max_von_mises"), "SF_yield": r.metrics.get("safety_factor_yield"),
                                     "deflection_mm": r.metrics.get("max_displacement")} for k, r in ch_results.items()}).T,
    pd.DataFrame({f"pins — {k}": {"load": f"{v['resultant_N']:.0f} N resultant", "max_von_mises_MPa": max(v["axle_bending_MPa"], v["pivot_pin_bending_MPa"]), "SF_yield": v["pin_SF"],
                                  "deflection_mm": float("nan")} for k, v in pins.items()}).T])
fig, ax = plt.subplots(figsize=(9, 4.5))
static["SF_yield"].astype(float).plot.barh(ax=ax, color=["#1565c0" if i.startswith("arm") else "#2e7d32" if i.startswith("chassis") else "#6a1b9a" for i in static.index])
ax.axvline(2.0, color="#c62828", ls="--", label="SF 2 (printed parts)"); ax.set(xlabel="safety factor to yield", xscale="log"); ax.legend(); ax.grid(alpha=0.3, which="both"); fig.tight_layout()
static.round(2)

## 4. Arm modes with the wheel on it, against the terrain excitation band

Terrain excites the wheel at frequencies up to `speed / shortest wavelength` (≈ 1.5 m/s / 0.125 m
= 12 Hz for the ISO band used) plus the wheel-hop mode of the quarter car; the arm's own bending mode
must sit well above both, or the arm rings on every rock.

The frequency diagram below is the rover's Campbell diagram: the excitations scale with speed instead of rpm — the wheel's own revolution (one and two per turn: a tyre joint, an out-of-round wheel), the pulse of crossing a rock of the terrain's typical size, and the broadband band of the road profile — against the arm modes, body bounce and wheel hop. No propeller, so no noise model: the terrain forces are the vibration source here.

In [ ]:
arm_modal = arm_model([], "arm_modal", masses=[talos.PointMass("axle", WHEEL_MASS_T)])
arm_modal.mesh(RUNS / "arm_modal")
modes = arm_modal.solve_modes(RUNS / "arm_modal", n_modes=4)
f_hop = math.sqrt((K_TYRE + K_SUSP) / M_WHEEL) / (2 * math.pi)
f_body = math.sqrt(K_SUSP / M_SPRUNG_CORNER) / (2 * math.pi)
print(f"arm modes with the wheel: {[round(f, 1) for f in modes.metrics['frequencies_hz']]} Hz | body bounce {f_body:.1f} Hz | wheel hop {f_hop:.1f} Hz | terrain band up to {1.5 / 0.125:.0f} Hz")
structure = chronos.Structure(tuple(modes.metrics["frequencies_hz"]), damping_ratio=0.04)
tviz.show(tviz.plot_mode(modes, mode=1))

In [ ]:
D_WHEEL = p["wheel_diameter"] / 1000
speeds = np.linspace(0.2, 3.0, 30)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(speeds, speeds / (math.pi * D_WHEEL), label="wheel 1/rev")
ax.plot(speeds, 2 * speeds / (math.pi * D_WHEEL), label="wheel 2/rev")
for name, tr in terrains.items():
    if tr["rocks"]:
        ax.plot(speeds, speeds / tr["rocks"][1], "--", label=f"rock crossing pulse, {name} ({tr['rocks'][1] * 100:.0f} cm rocks)")
ax.fill_between(speeds, speeds * 0.02, speeds * 8.0, color="#999", alpha=0.12, label="road profile band (ISO 8608, 0.02–8 cycles/m)")
for i, f in enumerate(modes.metrics["frequencies_hz"]):
    ax.axhline(f, color="#c62828", ls="--", lw=0.8); ax.text(speeds[0], f, f" arm mode {i + 1}: {f:.0f} Hz", color="#c62828", va="bottom", fontsize=8)
ax.axhline(f_body, color="#1565c0", ls="-.", lw=0.8); ax.text(speeds[-1], f_body, f"body bounce {f_body:.1f} Hz ", color="#1565c0", ha="right", va="bottom", fontsize=8)
ax.axhline(f_hop, color="#2e7d32", ls="-.", lw=0.8); ax.text(speeds[-1], f_hop, f"wheel hop {f_hop:.1f} Hz ", color="#2e7d32", ha="right", va="bottom", fontsize=8)
for name, s in sims.items():
    ax.axvline(s["speed"], color="#888", ls=":"); ax.text(s["speed"], 0.3, f" {name}", rotation=90, va="bottom", fontsize=8)
ax.set(xlabel="speed [m/s]", ylabel="frequency [Hz]", yscale="log", ylim=(0.2, 2000), title="frequency diagram: terrain and wheel excitations vs the rover's modes")
ax.legend(fontsize=7, loc="upper left"); ax.grid(alpha=0.3, which="both")
pd.DataFrame({name: {"speed_m_s": s["speed"], "wheel_1_rev_hz": s["speed"] / (math.pi * D_WHEEL),
                     "rock_pulse_hz": (s["speed"] / terrains[name]["rocks"][1]) if terrains[name]["rocks"] else float("nan"),
                     "nearest_arm_mode_hz": structure.nearest_mode(s["speed"] / (math.pi * D_WHEEL)),
                     "margin_wheel_to_arm_mode": structure.margin(s["speed"] / (math.pi * D_WHEEL))} for name, s in sims.items()}).T.round(2)

## 5. Cyclic loads: rainflow on the wheel-force histories (Chronos), one spectrum per terrain

The simulated histories are counted directly (ASTM rainflow) into blocks of the `wheel_z` pattern
(tyre force on the axle) and `wheel_x` (rock strikes). The time simulated is a few minutes; the blocks
are scaled to a mission of the stated duration.

In [ ]:
MISSION_MIN = {"paved patrol": 40.0, "gravel trail": 30.0, "rocky field": 20.0}
spectra = {}
for name, s in sims.items():
    scale = MISSION_MIN[name] * 60 / s["duration_s"]
    blocks = []
    for pattern, series in (("wheel_z", s["Ft"]), ("wheel_x", s["Fx"])):
        merged = {}
        for rng_, mean, count in chronos.rainflow(series[::4]):                  # 2 kHz -> 500 Hz, ample for < 30 Hz content
            if rng_ < 0.5:                                                        # ignore sub-0.5 N noise
                continue
            key = (round(mean, 1), round(rng_ / 2, 1))
            merged[key] = merged.get(key, 0.0) + count * scale
        blocks += [chronos.Block(pattern, m, a, c, f"{name}: {pattern} rainflow") for (m, a), c in merged.items()]
    spectra[name] = chronos.LoadSpectrum(name, MISSION_MIN[name] * 60, blocks, ["wheel_z", "wheel_x"],
                                         notes=f"quarter-car over {s['length']:.0f} m at {s['speed']} m/s, scaled to {MISSION_MIN[name]:.0f} min")
    spectra[name].save(RUNS / f"spectrum_{name.replace(' ', '_')}.json")
    fig = spectra[name].plot()
pd.DataFrame({k: {"blocks": len(sp.blocks), "cycles_per_mission": sp.total_cycles, "max_amplitude_N": max(b.amplitude for b in sp.blocks)} for k, sp in spectra.items()}).T.round(1)

## 6. Fatigue of the arm, damage map, life over a fleet usage

Unit cases: 100 N vertical and 100 N longitudinal at the axle. S-N for printed PA12-CF — **assumed**:
σ_f = 110 MPa, b = −0.10, Goodman with 90 MPa ultimate (flat orientation; across layers use less).

In [ ]:
unit_models = {"wheel_z": (arm_model([talos.Force("axle", fz=100.0)], "unit_z"), 100.0),
               "wheel_x": (arm_model([talos.Force("axle", fx=100.0)], "unit_x"), 100.0)}
unit_cases = {}
for k, (m, load) in unit_models.items():
    m.mesh(RUNS / k); unit_cases[k] = (m.solve(RUNS / k), load)
    print(k, unit_cases[k][0].status, round(unit_cases[k][0].metrics["max_von_mises"], 2), "MPa per 100 N")
CURVE = talos.FatigueCurve("PA12-CF (assumed)", sigma_f=110.0, b=-0.10, ultimate=90.0, source="assumed; coupon tests needed")
fatigue = {k: talos.assess_fatigue(unit_cases, sp.to_dict(), CURVE, workdir=RUNS / f"fatigue_{k.replace(' ', '_')}") for k, sp in spectra.items()}
life = pd.DataFrame({k: {"mission_min": MISSION_MIN[k], "damage_per_mission": f.result.metrics["damage_per_pass"],
                         "missions_to_failure": f.result.metrics["passes_to_failure"], "hours_to_failure": f.result.metrics["hours_to_failure"]}
                     for k, f in fatigue.items()}).T
life

In [ ]:
worst = life["damage_per_mission"].astype(float).idxmax()
tviz.show(tviz.plot_damage(fatigue[worst], unit_cases["wheel_z"][0].artifacts["mesh"]))

In [ ]:
damage = {k: f.result.metrics["damage_per_pass"] for k, f in fatigue.items()}
hours = {k: MISSION_MIN[k] / 60 for k in sims}
usage = {"paved patrol": 0.4, "gravel trail": 0.4, "rocky field": 0.2}
rate = sum(usage[k] * damage[k] for k in usage) / sum(usage[k] * hours[k] for k in usage)
sim = chronos.simulate_life(damage, hours, usage, n_flights=5000, seed=0)
fig = sim.plot()
print(f"usage {usage}: damage per 1000 h {rate * 1000:.3g} -> " + (f"{1 / rate:.0f} h to failure" if rate > 1e-9 else "not life-limiting (> 1e9 h)"))
mixes = {"as above": usage, "rocky-only": {"paved patrol": 0.0, "gravel trail": 0.0, "rocky field": 1.0}, "paved-only": {"paved patrol": 1.0, "gravel trail": 0.0, "rocky field": 0.0}}
pd.DataFrame({n: {"damage per 1000 h": sum(m[k] * damage[k] for k in m) / sum(m[k] * hours[k] for k in m) * 1000} for n, m in mixes.items()}).T

## 7. Print the arm (Mellonia)

Flat on its side (bores vertical): the layers then run along the arm, in the direction of the
bending stress, which is what the flat-orientation material values assume.

In [ ]:
prn = mellonia.slice_stl(cad["arm"].artifacts["stl"], GENERIC_PLA_0_2MM, mellonia.Orientation(rotate_x=90), RUNS / "print_arm")
print(prn)
if prn.ok:
    mviz.show(mviz.plot_toolpath(prn))

## 8. Export

In [ ]:
summary_doc = {"design": {"file": "designs/rover.py", "parameters": p}, "mass_kg": M_TOTAL,
               "suspension": {"k_susp_N_m": K_SUSP, "zeta": ZETA, "k_tyre_N_m": K_TYRE, "m_wheel_kg": M_WHEEL},
               "terrains": {k: {kk: v for kk, v in tr.items()} for k, tr in terrains.items()},
               "dynamics": summary.to_dict(), "arm_modes_hz": modes.metrics["frequencies_hz"],
               "operating_cases": {"grade_deg": GRADE_DEG, "mu_mud": MU_MUD, "roll_mud": ROLL_MUD, "payload_kg": PAYLOAD_KG, "h_cg_m": H_CG, "t_stall_Nm": T_STALL, "hill": hill, "hill_loaded": hill_loaded, "mud": mud, "mud_loaded": mud_loaded},
               "static": static.to_dict(orient="index"),
               "curve": CURVE.__dict__, "damage_per_mission": damage, "hours_per_mission": hours, "usage": usage,
               "damage_per_1000h": rate * 1000, "spectra": {k: str(RUNS / f"spectrum_{k.replace(' ', '_')}.json") for k in spectra}}
(RUNS / "rover_mechanics.json").write_text(json.dumps(summary_doc, indent=2, default=float))
print("written:", sorted(x.name for x in RUNS.iterdir()))

**Next steps an engineer would take:** measure the real spring rate and tyre stiffness (they set the
peak loads more than anything in the CAD); add the motor torque reaction and cornering loads to the
arm; run the chassis drop case with the battery mass; and, if the rocky-only column is the limiting
one, thicken the arm root or add a fillet at the pivot boss (the hotspot in the damage map) — a
revision in a `vegeta.core` workspace, or a campaign with `damage_per_1000h` as the objective.